# Olaverse — Language Detection Model Training

This notebook contains the complete workflow for training the Naive Bayes language detection model for Nigerian languages (Yoruba, Igbo, Hausa, Pidgin) and English.

## Methodology
1. **Corpus Definition**: We use representative sentences for Yoruba (`yor`), Igbo (`ibo`), Hausa (`hau`), Nigerian Pidgin (`pcm`), and English (`eng`).
2. **Feature Extraction**: We extract character-level n-grams of lengths 1, 2, 3, and 4.
3. **Model Selection**: A Multinomial Naive Bayes classifier is implemented with Laplace smoothing.
4. **Serialization**: We export the learned priors and conditional probabilities to a lightweight JSON file.

In [ ]:
import os
import json
import math
import re
from collections import defaultdict

## Define Training Corpus

In [ ]:
CORPUS = {
    "yor": [
        "Bawo ni, se daadaa ni?",
        "E kaaro, e ku ojumo.",
        "Ojo lo si oja lopo lopo lana.",
        "Kilo nsele ni adugbo yi?",
        "Oruko mi ni Olumide.",
        "Mo feran lati je iyan ati egusi.",
        "Gbagbe nipa oro yen.",
        "Se o ti jeun?",
        "Olorun bukun fun o.",
        "Ẹ kú àbọ̀ ooo.",
        "A dupe lowo yin fun iranlowo yin.",
        "Bawo ni oro na se ri?",
        "Mo n lo si ile-iwe.",
        "Eṣe pupo fun ore yin.",
        "Ṣe alafia ni o wa?",
        "Ọjọ́ àìkú jẹ́ ọjọ́ ìsinmi.",
        "Alafia fun ile yi.",
        "Ki ni oruko re?",
        "Nibo ni o n gbe?",
        "E je ki a lo si ile."
    ],
    "ibo": [
        "Kedu ka i mere?",
        "O tege anyi siri fukwa.",
        "Aha m bu Chidi.",
        "Ijeoma, nọrọ nke ọma.",
        "Kedu aha gị?",
        "Ego ole ka ihe a bụ?",
        "Nri a dị ezigbo mma.",
        "Bia ebe a nwa m.",
        "Chineke gọzie gị.",
        "Ihe a masịrị m nke ukwuu.",
        "Kedu ka ọ dị?",
        "Ọ dị mma, nna m.",
        "Ije ọma na njem gị.",
        "Kedu ka ezinụlọ gị dị?",
        "Ebee ka ị na-aga?",
        "Kedụ ka ị mere taa?",
        "Ị dị njikere?",
        "Ahụrụ m gị n'anya.",
        "Ndị a bụ ezinụlọ m.",
        "Ego ole ka ị nwere?"
    ],
    "hau": [
        "Ina kwana? Yaya gida?",
        "Sannu da zuwa.",
        "Ina kwana? Lafiya lau.",
        "Yaya kake? Sannu da aiki.",
        "Ina son wannan abincin.",
        "Gida na yana da kyau.",
        "Mungode kwarai da gaske.",
        "Yaya iyali?",
        "Ina ne kasuwa take?",
        "Ruwa yana da dumi.",
        "Sannu aboki na.",
        "Muna son kasar mu Nijeriya.",
        "Wannan fim din yana da kyau sosai.",
        "Yaya kwanan gida?",
        "Ina son in koyi Hausa.",
        "Zan je gida gobe.",
        "Ina jin yunwa yanzu.",
        "Wane ne sunanka?",
        "Ina kake da zama?",
        "Barka da yamma abokina."
    ],
    "pcm": [
        "How far, wetin dey happen?",
        "I no like am at all, e too bad.",
        "This film too sweet, abeg.",
        "Wetin you dey chop?",
        "Na true you talk, no cap.",
        "E get as e be, sha.",
        "No wahala, we go see later.",
        "Make we waka comot for here.",
        "I don chop belly full.",
        "Shey you dey hear me so?",
        "How body? De work dey go well?",
        "No mind am, na mumu post.",
        "E dey talk wetin he no sabi.",
        "Abeg no dey do shakara for here.",
        "I get correct gist for you.",
        "Make you no go dey do mumu thing.",
        "Wetin concern me inside?",
        "Carry your wahala comot.",
        "Awuf dey run belly, abeg.",
        "You don see my pikin?"
    ],
    "eng": [
        "How are you doing today?",
        "Good morning, how was your night?",
        "What is happening around here?",
        "My name is John.",
        "I love to eat rice and chicken.",
        "Forget about that issue.",
        "Have you eaten your lunch?",
        "God bless you.",
        "Welcome back home.",
        "Thank you very much for your help.",
        "How does the issue look?",
        "I am going to school.",
        "Thank you very much for your friendship.",
        "Are you in good health?",
        "Sunday is a day of rest.",
        "Peace be unto this house.",
        "What is your name?",
        "Where do you live?",
        "Let us go home.",
        "This movie is very good indeed."
    ]
}

## Define N-gram Extraction

In [ ]:
def extract_ngrams(text, n_min=1, n_max=4):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    padded = f"_{text}_"
    ngrams = []
    for n in range(n_min, n_max + 1):
        for i in range(len(padded) - n + 1):
            ngrams.append(padded[i:i+n])
    return ngrams

## Model Training and Export

In [ ]:
languages = list(CORPUS.keys())
total_docs = sum(len(docs) for docs in CORPUS.values())
priors = {lang: math.log(len(docs) / total_docs) for lang, docs in CORPUS.items()}

ngram_counts = defaultdict(lambda: defaultdict(int))
lang_totals = defaultdict(int)
vocab = set()

for lang, docs in CORPUS.items():
    for doc in docs:
        ngrams = extract_ngrams(doc)
        for ngram in ngrams:
            ngram_counts[ngram][lang] += 1
            lang_totals[lang] += 1
            vocab.add(ngram)

sorted_features = sorted(vocab, key=lambda ng: sum(ngram_counts[ng].values()), reverse=True)
selected_features = set(sorted_features[:2000])

alpha = 0.1
vocab_size = len(selected_features)

features_probs = {}
for ngram in selected_features:
    features_probs[ngram] = {}
    for lang in languages:
        count = ngram_counts[ngram][lang]
        prob = (count + alpha) / (lang_totals[lang] + alpha * vocab_size)
        features_probs[ngram][lang] = math.log(prob)

default_probs = {}
for lang in languages:
    prob = alpha / (lang_totals[lang] + alpha * vocab_size)
    default_probs[lang] = math.log(prob)

default_log_prob = sum(default_probs.values()) / len(default_probs)

model = {
    "priors": priors,
    "features": features_probs,
    "default_log_prob": default_log_prob
}

# Save the model weights
model_dir = "../../olaverse/models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "language_detector.json")
with open(model_path, "w", encoding="utf-8") as f:
    json.dump(model, f, indent=2)
print(f"Model successfully exported to {model_path}")

## Evaluate Classifier

In [ ]:
def predict(text):
    ngrams = extract_ngrams(text)
    scores = {lang: priors[lang] for lang in languages}
    for ngram in ngrams:
        if ngram in features_probs:
            for lang in scores:
                scores[lang] += features_probs[ngram][lang]
        else:
            for lang in scores:
                scores[lang] += default_log_prob
    return max(scores, key=scores.get)

# Test cases
tests = [
    ("Bawo ni, se daadaa ni?", "yor"),
    ("Ina kwana?", "hau"),
    ("Kedu ka ị mere?", "ibo"),
    ("How far, wetin dey happen?", "pcm"),
    ("This movie is very good indeed.", "eng")
]

correct = 0
for text, expected in tests:
    pred = predict(text)
    is_correct = pred == expected
    if is_correct:
        correct += 1
    print(f"Text: '{text}' | Expected: {expected} | Pred: {pred} | {'✅' if is_correct else '❌'}")

print(f"Accuracy: {correct / len(tests) * 100}%")